In [1]:
!pip install gradio opencv-python gtts requests playsound

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 7.3 MB/s eta 0:00:00
  Created wheel for playsound: filename=playsound-1.3.0-py3-none-any.whl size=7020 sha256=101fef510d711b5186cd12f0dfef0eee1ad8d272a1eb6404cc3f7b7af65a277b
  Stored in directory: /root/.cache/pip/wheels/cf/42/ff/7c587bae55eec67b909ca316b250d9b4daedbf272a3cbeb907
Successfully built playsound
  Attempting uninstall: click
    Found existing installation: click 8.2.1
    Uninstalling click-8.2.1:
      Successfully uninstalled click-8.2.1


In [ ]:
import gradio as gr
import base64
import requests
from gtts import gTTS
import uuid
import os
import tempfile

# ===== CONFIG =====
GEMINI_API_KEY = 'Your_API_Key'  # Replace with your real key
GEMINI_MODEL = "gemini-2.0-flash"
GEMINI_VISION_URL = f'https://generativelanguage.googleapis.com/v1/models/{GEMINI_MODEL}:generateContent?key={GEMINI_API_KEY}'

# Convert image to base64
def image_to_base64(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

# Get AI description using Gemini
def get_image_description(image_base64):
    headers = {
        "Content-Type": "application/json"
    }

    payload = {
        "contents": [
            {
                "parts": [
                    {
                        "inlineData": {
                            "mimeType": "image/jpeg",
                            "data": image_base64
                        }
                    },
                    {
                        "text": "Describe this image clearly and helpfully for a blind person make it short in one line."
                    }
                ]
            }
        ]
    }

    response = requests.post(GEMINI_VISION_URL, headers=headers, json=payload)

    if response.status_code == 200:
        try:
            return response.json()['candidates'][0]['content']['parts'][0]['text']
        except Exception as e:
            return f"Error parsing response: {str(e)}"
    else:
        return f"API Error: {response.status_code} - {response.text}"

# Generate TTS audio
def describe_and_generate_audio(image_path):
    if image_path is None:
        return "No image uploaded.", None

    image_base64 = image_to_base64(image_path)
    description = get_image_description(image_base64)

    # Save audio
    tts = gTTS(description)
    audio_path = os.path.join(tempfile.gettempdir(), f"{uuid.uuid4().hex}.mp3")
    tts.save(audio_path)

    return description, audio_path

# Gradio UI
iface = gr.Interface(
    fn=describe_and_generate_audio,
    inputs=gr.Image(type="filepath", label="Upload an image"),
    outputs=[
        gr.Textbox(label="AI Description"),
        gr.Audio(label="Audio Description", type="filepath")
    ],
    title="AI Image Description for the Blind",
    description=""
)

iface.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://30d152e5c3eed3c76f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
